# MJO 3: Calculate EOFs

Via `mjo_EOF.ncl`

Calculating the empirical orthognal functions (EOFs)

- [eofunc_eofs](https://geocat-comp.readthedocs.io/en/v2023.02.0/user_api/generated/geocat.comp.stats.eofunc_eofs.html): computes the empirical orthogonal functions (EOFs, aka: Principal Component Analysis. Python alterative to NCL function `eofunc`, built on [`eofs`](https://ajdawson.github.io/eofs/) package
- [geocat-example: NCL_eof_1_1.py](https://geocat-examples.readthedocs.io/en/latest/gallery/Contours/NCL_eof_1_1.html)

In [1]:
# # getenv == os.environ
import os

CASENAME = "QBOi.EXP1.AMIP.001"
startdate = '19790101'
enddate = '19811231'

# /glade/u/home/bundy/mdtf/MDTF_3_main/MDTF-diagnostics.blocking_notebook/diagnostics/MJO_suite/MJO_driver.py
#DATADIR = "/glade/u/home/bundy/diag/mdtf/inputdata/model/QBOi.EXP1.AMIP.001/"
WORK_DIR = os.getcwd()
DATADIR = "/data/"

U200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U200.day.nc"
V200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V200.day.nc"
U850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U850.day.nc"
V850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V850.day.nc"
RLUT_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.FLUT.day.nc"
PR_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.PRECT.day.nc"

lev_coord = "lev"
lat_coord = "lat"
lon_coord = "lon"
time_coord = "time"
pr_var = "PRECT"
rlut_var = "FLUT"

u200_var = "U200"
v200_var = "V200"

wk_dir = WORK_DIR + "/model/"

# setup data directory, if does not already exist
#if not os.path.exists(DATADIR): os.makedirs(DATADIR)

In [2]:
# input data
anom_dir = WORK_DIR + DATADIR + "anomaly/"
anom_files = [file for file in os.listdir(anom_dir) if "anom" in file]
anom_files

['QBOi.EXP1.AMIP.001.u200.day.anom.nc',
 'QBOi.EXP1.AMIP.001.prect.day.anom.nc',
 'QBOi.EXP1.AMIP.001.v200.day.anom.nc',
 'QBOi.EXP1.AMIP.001.v850.day.anom.nc',
 'QBOi.EXP1.AMIP.001.flut.day.anom.nc',
 'QBOi.EXP1.AMIP.001.u850.day.anom.nc']

In [3]:
# for example: U200
import xarray as xr
xr.open_dataset(anom_dir + anom_files[0])

<xarray.Dataset> Size: 565MB
Dimensions:  (time: 2555, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 20kB ...
    U200     (time, lat, lon) float32 565MB ...

In [29]:
# directory for plots

pltDir = WORK_DIR + "/PS/"
pltType = "ps"

print(pltDir)
if not os.path.exists(pltDir): os.makedirs(pltDir)

/home/cs/Github/geocat-research/mjo/PS/


In [30]:
# EOF variables
seasons_names = ["winter", "summer"]

neof = 4
latS = -30
latN = 30

In [31]:
# calculate Guassian weights

import xarray as xr
import numpy as np

file_list = [PR_FILE, RLUT_FILE, U200_FILE, U850_FILE, V200_FILE, V850_FILE]

# originally set up in daily_netcdf.ncl (TODO)
#latS = -40
#latN = 40

pr_xarray = xr.open_dataset(anom_dir + anom_files[0])

# calculate the guassian weights of the first file
gw0 = np.abs(np.sin(pr_xarray.lat * np.pi / 180).diff("lat"))
gw0 = gw0.sel(lat=slice(latS, latN))
gw0

<xarray.DataArray 'lat' (lat: 64)> Size: 512B
array([0.01422174, 0.01435572, 0.01448582, 0.01461199, 0.01473422,
       0.01485246, 0.01496668, 0.01507685, 0.01518295, 0.01528493,
       0.01538278, 0.01547647, 0.01556597, 0.01565126, 0.01573232,
       0.01580912, 0.01588164, 0.01594987, 0.01601378, 0.01607336,
       0.01612859, 0.01617946, 0.01622595, 0.01626805, 0.01630575,
       0.01633904, 0.01636791, 0.01639235, 0.01641236, 0.01642792,
       0.01643904, 0.01644572, 0.01644794, 0.01644572, 0.01643904,
       0.01642792, 0.01641236, 0.01639235, 0.01636791, 0.01633904,
       0.01630575, 0.01626805, 0.01622595, 0.01617946, 0.01612859,
       0.01607336, 0.01601378, 0.01594987, 0.01588164, 0.01580912,
       0.01573232, 0.01565126, 0.01556597, 0.01547647, 0.01538278,
       0.01528493, 0.01518295, 0.01507685, 0.01496668, 0.01485246,
       0.01473422, 0.01461199, 0.01448582, 0.01435572])
Coordinates:
  * lat      (lat) float64 512B -29.69 -28.74 -27.8 -26.86 ... 27.8 28.74 29.69
Attributes:
    units:      degrees_north
    long_name:  latitude

### Fixed Lanczos Filter Weights

Create bandpass filter from weights

See: [Lanczos Filter Weights](https://www.ncl.ucar.edu/Applications/mjoclivar.shtml)

```
When needed, the weights for the suggested 20-100 day bandpass Lanczos filter are generated 'on-the-fly' using:

  ihp      = 2                             ; bpf=>band pass filter
  nWgt     = 201
  sigma    = 1.0                           ; Lanczos sigma
  fca      = 1./100.
  fcb      = 1./20.
  wgt      = filwgts_lanczos (nWgt, ihp, fca, fcb, sigma )

```

See `generate_lanczos_filter_weights.ncl` which generates `lanczos_filter_weights_output.txt`

In [32]:
import numpy as np
def lanczos_filter_weights(nwt=None, ihp=None, fca=None, fcb=None, nsigma=None):
    # nwt = scale indicating the total number of weights (must be an odd number, nwt >= 3).The more weights, the better the filter, but greater loss of data
    # ihp = scale indicating the low-pass filter
    # fca = scale indicating the cut-off frequency of the ideal high or low-pass filter (0 < fca < 0.5)
    # fcb = scale used only when band-pass filter is desired, second cut-off frequency (fca < fcb < 0.5)
    # scale indicating the power of sigma factor (nsigma >= 0), ngima = 1 is common
    # returns: a symmetrical set of weights
    ncl_output_weights = np.loadtxt("lanczos_filter_weights_output.txt", delimiter=",", skiprows=17)
    return ncl_output_weights 

In [33]:
## calculate one-dimensional filter weights (filwgts_lanczos) 

# create BandPass filter
#ihp = 2 # bpf->band pass filter, 2 = band-pass
#nWgt = 201 # number of weights (must be an odd number, the more weights, the better the filter, but the greater loss of data)
#sigma = 1 # lanczos sigma
#fca = 1/100 # indicating the cut-off frequency of the ideal high/low-pass filter (0.0 < fca < 0.5)
#fcb = 1/20 # scalar used when band-pass filter is desired. It is the second cut-off frequency  (fca < fcb < 0.5)
weights = lanczos_filter_weights()
weights

array([ 1.933179e-11, -1.602059e-05, -4.602891e-05, -8.413403e-05,
       -1.211932e-04, -1.459612e-04, -1.466584e-04, -1.127676e-04,
       -3.682710e-05,  8.402883e-05,  2.470150e-04,  4.433134e-04,
        6.584777e-04,  8.736432e-04,  1.067462e-03,  1.218581e-03,
        1.308390e-03,  1.323730e-03,  1.259200e-03,  1.118744e-03,
        9.162520e-04,  6.749944e-04,  4.258647e-04,  2.045088e-04,
        4.758891e-05, -1.146049e-05,  5.326822e-05,  2.562978e-04,
        5.978383e-04,  1.062210e-03,  1.617924e-03,  2.219547e-03,
        2.811319e-03,  3.332239e-03,  3.722208e-03,  3.928611e-03,
        3.912660e-03,  3.654768e-03,  3.158300e-03,  2.451133e-03,
        1.584706e-03,  6.304432e-04, -3.262688e-04, -1.194086e-03,
       -1.884996e-03, -2.323995e-03, -2.458149e-03, -2.264029e-03,
       -1.752608e-03, -9.709108e-04, -1.243057e-09,  1.050766e-03,
        2.052896e-03,  2.870605e-03,  3.374352e-03,  3.454755e-03,
        3.035499e-03,  2.083817e-03,  6.173067e-04, -1.293891e

In [82]:
from datetime import datetime
import xeofs as xe

for anom in anom_files:
    # read anomalies
    ivars = anom.split(".")[4].upper()
    print(f"Anomaly File: {anom_dir + anom}")
    print(f"Starting: {ivars}") # L91 (mjo_eof.ncl)

    # read anomalies
    anom_netcdf = xr.open_dataset(anom_dir + anom)
    X = anom_netcdf[ivars].sel(lat=slice(latS, latN)) # L96 (mjo_EOF.ncl)
    time = anom_netcdf[ivars].time # L101 (mjo_EOF.ncl)
    #print(time[0].dt.month.values)

    yrStrt = time.min().item() # L104 (mjo_EOF.ncl)
    yrLast = time.max().item() # L105 (mjo_EOF.ncl)

    lat = X.lat
    lon = X.lon
    gw = gw0.sel(lat=slice(latS, latN))
    mlon = len(lon)
    nlat = len(lat)

    # apply the band pass filter to the original anomalies
    # wgt_runave_Wrap(): calculates a weighted average on the rightmost dimension and retain metadata
    # wgt_runave_Wrap(X(lat|:, lon|:, time|:), wgt, 0)
    reordered_x = X.transpose("lat", "lon", "time") # reorder array from (time, lat, lon) to (lat, lon, time)
    # weighted mean = wgt_areaave_Wrap()
    wgts = xr.DataArray(weights, dims=["weights"]) # convert weights to DataArray
    weighted_data = reordered_x.weighted(wgts) # weighted mean
    x = weighted_data.mean(("lat", "lon")) # area average

    # remove means of band pass eries (not necessary)
    x -= x.mean(dim="time") # L126 (mjo_EOF.ncl)

    # two seasons, Nov to Apr & May to October
    count = np.zeros(2, dtype="int") # L131 (mjo_EOF.ncl)
    for n in range(len(time)):
        if (time[n].dt.month.values >= 5) and (time[n].dt.month.values <= 5):
            count[1] += 1
        else:
            count[0] += 1

    time0 = np.zeros(count[0], dtype="double") # L141 (mjo_EOF.ncl)
    time1 = np.zeros(count[1], dtype="double") # L142 (mjo_EOF.ncl)

    if len(time) > 1000:
        time_warning = (f"{len(time)} time samples, may take a while, check {pltDir}")
    else:
        time_warning = (f"no time warning, nt {len(time)}")

    for season in [0, 1]:
        pltName = f"{CASENAME}.MJO.EOF.{ivars}.{seasons_names[season]}" # L152 (mjo_EOF.ncl)
        print(pltName)
        print(f"\nComputing {ivars}, {seasons_names[season]}. {time_warning}")
        x_season = np.zeros((count[season], nlat, mlon), dtype="float")
        x_season = xr.DataArray(x_season,
                                dims=("time", "lat", "lon"),
                                coords={"lat": lat, "lon": lon})
        if season == 0:
            index = 0
            for n in range(len(time)):
                if (time[n].dt.month.values <= 4) or (time[n].dt.month.values >= 11):
                    time0[index] = np.array(time[n], dtype="datetime64[D]")
                    print(x_season)
                    print(x)
                    x_season[index] = x.sel(time=time[n])
                    print(x_season)
                    break
        break
    break

Anomaly File: /home/cs/Github/geocat-research/mjo/data/anomaly/QBOi.EXP1.AMIP.001.u200.day.anom.nc
Starting: U200
QBOi.EXP1.AMIP.001.MJO.EOF.U200.winter

Computing U200, winter. 2555 time samples, may take a while, check /home/cs/Github/geocat-research/mjo/PS/
<xarray.DataArray (time: 2338, lat: 64, lon: 288)> Size: 345MB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
...
        ...,
        [0., 0., 0., ..., 0., 0

ValueError: new dimensions ('lat', 'lon') must be a superset of existing dimensions ('weights',)